<a href="https://colab.research.google.com/github/DanielV-13/FNPE---CALCULO-DE-PARAMETROS-de-MODELO-DE-LA-GOTA-LIQUIDA-SEMF-/blob/main/FNPE_MODELO_DE_LA_GOTA_LIQUIDA_(SEMF).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica Computacional: Modelo de la Gota Líquida y Fórmula Semiempírica de Masas (SEMF)

## 📌 Contexto General y Objetivos
En esta práctica de **Física Nuclear y de Partículas**, analizamos las propiedades macroscópicas y microscópicas de los núcleos atómicos utilizando un enfoque semiclásico conocido como el **Modelo de la Gota Líquida** (propuesto por George Gamow y desarrollado por Carl Friedrich von Weizsäcker)[cite: 18].

El objetivo principal es comprobar cómo una serie de términos físicos e interacciones fundamentales (la fuerza nuclear fuerte, la repulsión culombiana, el Principio de Exclusión de Pauli y el apareamiento de espines) logran explicar con gran precisión la estabilidad y la energía de enlace de los núcleos atómicos[cite: 18, 21].

## 📂 Origen de los Datos
* **Fuente Oficial:** Portal del *National Nuclear Data Center* (NNDC), utilizando la base de datos de las **Nuclear Wallet Cards** (`https://www.nndc.bnl.gov/walletcards/search.html`)[cite: 2, 11].
* **Variables extraídas del archivo (`walletcards.csv`):**
  * `Atomic Number (Z)`: Número de protones (define el elemento químico).
  * `Atomic Mass (A)`: Número total de nucleones (protones + neutrones).
  * `Mass Excess`: Exceso de masa experimental necesario para calcular las energías de enlace reales.
  * `Half-Life`: Vida media del nucleido, utilizada para filtrar y clasificar los núcleos en **estables** (`"STABLE"`) e **inestables**.

In [2]:
# Importación de librerías esenciales para análisis numérico y visualización
import numpy as np   #Numpy
import pandas as pd  #Pandas
import matplotlib.pyplot as plt  #Matplotlib

# 1. URL RAW directa de GitHub
url_github = "https://raw.githubusercontent.com/DanielV-13/FNPE---CALCULO-DE-PARAMETROS-de-MODELO-DE-LA-GOTA-LIQUIDA-SEMF-/refs/heads/main/walletcards.csv"

# 2. Leer el CSV manejando posibles líneas irregulares
try:
    df = pd.read_csv(url_github)
except Exception as e:
    # Si por alguna razón el delimitador es diferente (ej. tabulaciones), probamos con sep=None y engine='python'
    df = pd.read_csv(url_github, sep=None, engine='python')

print("¡Datos descargados y leídos exitosamente desde GitHub!")

# 3. Filtrar únicamente los estados fundamentales (Level Index == 0)
df_base = df[df['Level Index'] == 0].copy()

# 4. Identificar núcleos estables basados en la columna Half-Life
df_base['is_stable'] = df_base['Half-Life'].astype(str).str.upper() == 'STABLE'

print(f"Total de registros analizados en el estado fundamental: {len(df_base)}")
print(f"Núcleos estables identificados: {df_base['is_stable'].sum()}")
df_base.head()

¡Datos descargados y leídos exitosamente desde GitHub!
Total de registros analizados en el estado fundamental: 3371
Núcleos estables identificados: 253


,Atomic Number (Z),Atomic Mass (A),Level Index,Element,Level Energy (Modifier),Level Energy,Level Energy (Unit),Level Energy (Error),Spin-Parity,Half-Life (Modifier),...,Decay Width (Error),Abundance (Modifier),Abundance,Abundance (Error),Mass Excess,Mass Excess (Unit),Mass Excess (Error),Decay Modes,Unnamed: 24,is_stable
0,0,1,0,NN,NaN,0.0,keV,NaN,1/2+,NaN,...,NaN,NaN,NaN,NaN,8071.318060,keV,0.000440,B- = 100,NaN,False
1,1,1,0,H,NaN,0.0,keV,NaN,1/2+,NaN,...,NaN,NaN,99.9855,NaN,7288.971064,keV,0.000013,NaN,NaN,True
2,1,2,0,H,NaN,0.0,keV,NaN,1+,NaN,...,NaN,NaN,0.0145,NaN,13135.722895,keV,0.000015,NaN,NaN,True
3,1,3,0,H,NaN,0.0,keV,NaN,1/2+,NaN,...,NaN,NaN,NaN,NaN,14949.810900,keV,0.000080,B- = 100,NaN,False
4,1,4,0,H,NaN,0.0,keV,NaN,2-,NaN,...,0.3,NaN,NaN,NaN,24621.129000,keV,100.000000,n = 100,NaN,False


## ⚛️ Fundamento Físico: ¿Cómo calculamos la Energía de Enlace ($B$)?

Experimentalmente, un núcleo atómico pesa *menos* que la suma individual de sus protones y neutrones por separado. Esa "masa perdida" es la energía de enlace $B$ que mantiene unidas a las partículas mediante la **fuerza nuclear fuerte**:

$$B(Z, A) = [Z \cdot m_p + N \cdot m_n - M_{\text{núcleo}}] \cdot c^2$$

En las tablas de datos nucleares, este déficit o exceso de masa se cuantifica a través del **Exceso de Masa ($\Delta$)**. A partir de él, calculamos la energía de enlace total y la dividimos entre $A$ para obtener la **energía de enlace por nucleón ($B/A$)**, que es la métrica estándar de estabilidad en física nuclear.

## ⚛️ Paso 2: Cálculo de la Energía de Enlace por Nucleón ($B/A$)

A partir del exceso de masa (`Mass Excess`) reportado en las Nuclear Wallet Cards, calculamos la energía de enlace total $B$ mediante la diferencia de masa entre los nucleones libres (protones y neutrones) y el núcleo enlazado:

$$B(Z, A) = [Z \cdot m_p + N \cdot m_n - M_{\text{núcleo}}] \cdot c^2$$

Luego, dividimos entre el número total de nucleones $A$ para obtener $B/A$, la cual representa qué tan fuertemente ligado está cada nucleón en promedio.

In [ ]:
ASAS